# Automated Sarcopenia Assessment: L3 Muscle Segmentation via Deep Learning


In this initial phase, we are working on preparing the raw clinical dataset. The original medical images (CT scans and masks) are provided in compressed .nii.gz format, organized by patient folder. 

This section goes through all this arguments:
1. **Decompression and integration:** Estract the files and convert them in NIfTI .nii format. 
2. **Clinical anonimization:** Substitute the original folder names with anonimus ID (subject_X) to preserve the sensible data privacy, saving all in another file that locally maps for medical tracciability.
3. **Masks Sanity Check:** Verify that the masks files exclusively have the expected classifications $[0, 1, 2]$.
4. **Native risolution verification:** Inspect the original dimensionality of the TAC matrices.

## 1. Data Ingestion, Decompression & Anonymization Pipeline

Medical datasets often arrive in compressed formats (.nii.gz) and contain highly sensitive patient identifiers within their folder structures. 
To ensure compliance with clinical privacy standards and optimize training efficiency, this module executes three tasks:
1. **Decompression:** Extracts raw .nii.gz spatial volumes into uncompressed .nii  files for faster disk I/O.
2. **Anonymization:** Converts original patient names into a standardized format ( subject_0 ,  subject_1 , etc.).
3. **Traceability:** Generates an isolated, secure CSV mapping file ( mapping_patients.csv ) to link anonymous IDs back to original clinical records securely if clinical verification is needed.

In [3]:
# #Extraction of zip files 

# import shutil
# import gzip
# import csv
# from pathlib import Path

# # My directions
# percorso_base = Path(r"C:\Users\vella\Desktop\tirocinio\Dataset")
# cartella_sorgente = percorso_base / "Dataset"

# cartella_slices = percorso_base / "slices"
# cartella_masks = percorso_base / "masks"
# cartella_slices.mkdir(parents=True, exist_ok=True)
# cartella_masks.mkdir(parents=True, exist_ok=True)

# # CVS file for mapping patients 
# file_mapping = percorso_base / "mapping_pazienti.csv"

# pazienti_ok = 0
# lista_mapping = []

# print("Inizio estrazione e anonimizzazione dei file...\n")

# # iteration for patients counting
# for indice, cartella_paziente in enumerate(cartella_sorgente.iterdir(), start=0):
#     if cartella_paziente.is_dir():
#         nome_originale = cartella_paziente.name
        
#         # anonimus name 
#         nuovo_nome = f"subject_{indice}"
        
    
#         path_file_ct = cartella_paziente / "ct_slice.nii.gz"
#         path_file_mask = cartella_paziente / "mask_slice.nii.gz"
        
#         # final destination paths
#         dest_ct = cartella_slices / f"slice_{nuovo_nome}.nii"
#         dest_mask = cartella_masks / f"mask_{nuovo_nome}.nii"
        
#         if path_file_ct.exists() and path_file_mask.exists():
#             try:
#                 with gzip.open(path_file_ct, 'rb') as f_in:
#                     with open(dest_ct, 'wb') as f_out:
#                         shutil.copyfileobj(f_in, f_out)
                
#                 with gzip.open(path_file_mask, 'rb') as f_in:
#                     with open(dest_mask, 'wb') as f_out:
#                         shutil.copyfileobj(f_in, f_out)
                        
#                 #print(f" {nome_originale} -> convertito in -> {nuovo_nome}")
                
#                 # saving the mapping 
#                 lista_mapping.append([nuovo_nome, nome_originale])
#                 pazienti_ok += 1
                
#             except Exception as e:
#                 print(f" Errore durante l'estrazione di {nome_originale}: {e}")
#         else:
#             print(f" {nome_originale}: Impossibile trovare i file .nii.gz originali")

# # CVS file creation
# with open(file_mapping, mode='w', newline='', encoding='utf-8') as f_csv:
#     writer = csv.writer(f_csv)
#     writer.writerow(["ID_Anonimo", "Cartella_Originale"])
#     writer.writerows(lista_mapping)

# print(f" {pazienti_ok} pazienti pronti in formato subject_X.nii.")
# print(f"Mappa dei nomi salvata in: {file_mapping}")

## 2. Categorical Label Integrity & Quality Assurance (QA)

Before feeding the dataset into a multiclass segmentation model, we must run a strict sanity check on the semantic masks. 
For a 3-class U-Net targeting sarcopenia assessment, the masks must only contain the following unique integer pixel values:
* **0**: Background / Unclassified tissues.
* **1**: Abdominal Muscle Wall.
* **2**: Psoas Muscle.

Any outlier labels (e.g., decimal values, artifact labels, or missing classes) will corrupt the loss functions during training.

In [5]:
print(" Checking structural and categorical integrity of target segmentation masks...\n")

anomalies_count = 0
EXPECTED_LABELS = {0, 1, 2}

# Target processed masks
for mask_path in cartella_masks.glob("*.nii"):
    # Load raw voxel data
    mask_data = nib.load(mask_path).get_fdata()
    
    # Extract unique numerical values within this specific slice
    unique_values = np.unique(mask_data)
    actual_labels = set(unique_values)
    
    # Catch any deviations from standard mapping
    if actual_labels != EXPECTED_LABELS:
        print(f" Label Anomaly detected in {mask_path.name}: Found values {unique_values}")
        anomalies_count += 1

print("--------------------------------------------------")
if anomalies_count == 0:
    print("Integrity Check Passed: All masks contain strictly valid categorical values [0, 1, 2].")
else:
    print(f"Integrity Check Failed: Found {anomalies_count} anomalous mask files requiring remediation.")

 Checking structural and categorical integrity of target segmentation masks...

--------------------------------------------------
Integrity Check Passed: All masks contain strictly valid categorical values [0, 1, 2].


## 3. Spatial Dimensionality & Resolution Audit

Finally, we run a resolution check on a sample imaging slice. Knowing the native voxel array size allows us to verify our pipeline's input constraints before applying downsampling matrices inside the deep learning generator.

In [14]:
import nibabel as nib

# Sostituisci con il percorso reale di una tua immagine
percorso = r"C:\Users\vella\Desktop\tirocinio\Dataset\slices\slice_subject_3.nii"
percorso_2 = r"C:\Users\vella\Desktop\tirocinio\Dataset\slices\slice_subject_100.nii"
percorso_3 = r"C:\Users\vella\Desktop\tirocinio\Dataset\slices\slice_subject_50.nii"

print(f"Dimensioni: {nib.load(percorso).shape}")
print(f"Dimensioni: {nib.load(percorso_2).shape}")
print(f"Dimensioni: {nib.load(percorso_3).shape}")

Dimensioni: (512, 512, 1)
Dimensioni: (512, 512, 1)
Dimensioni: (512, 512, 1)


## Pipeline Architecture Overview

The data engineering process is structured into four sequential phases, ensuring data integrity, patient privacy, and mathematical stability before the tensors reach the Neural Network.

```text
 
                 PHASE 1: INGESTION & ANONYMIZATION
 
[ Raw .nii.gz Archives ] 
       │
       ├──> 1. Decompression (GZIP Extraction)
       ├──> 2. Patient De-identification (Subject_X)
       └──> 3. Secure CSV Mapping Generation
       │
       ▼
[ Standardized .nii Files (Slices & Masks) ]

 
                 PHASE 2: QUALITY ASSURANCE (QA)
 
       │
       ├──> 1. Categorical Integrity Audit (Strictly [0, 1, 2] labels)
       └──> 2. Spatial Resolution Check
       │
       ▼
[ Validated Medical Dataset ]


